# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(Complété pour LinkedIn Job Postings & Salaries)*
- **Titre du sujet :** Analyse du Marché de l'Emploi, des Compétences et des Rémunérations (LinkedIn)
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `arshkon/linkedin-job-postings` (Multi-tables relationnelles : offres d'emploi, entreprises, salaires, compétences)
- **Clé(s) de jointure envisagée(s) entre sources / tables :**
  - `job_id` (entre les tables `postings`, `salaries`, `job_skills` et `job_industries`)
  - `company_id` (entre `postings` et `companies`)
- **Problématique métier principale :**
  - Identifier la dynamique de recrutement et les niveaux de salaires proposés selon les compétences requises, les secteurs d'activité et la répartition géographique.
  - Déterminer les compétences clés à forte valeur ajoutée et analyser les tendances d'embauche des entreprises de différentes tailles.
- **Variables clés retenues :**
  - **Temporelles :** `original_listed_time`, `expiry`
  - **Quantitatives :** `med_salary`, `min_salary`, `max_salary`, `views`, `applies`
  - **Catégorielles :** `company_name`, `location`, `formatted_work_type`, `skill_abr`, `industry_name`


## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# Dataset Kaggle retenu pour le projet
KAGGLE_DATASETS: list[str] = [
    "arshkon/linkedin-job-postings",
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :\n")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")


Inventaire local multi-sources dans /workspace/data_raw (22 fichier(s)) :

 - linkedin-job-postings/companies/companies.csv             22.12 Mo  (~2 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/companies/company_industries.csv     0.75 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/companies/company_specialities.csv    4.23 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/companies/employee_counts.csv        1.00 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/jobs/benefits.csv                    1.84 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/jobs/job_industries.csv              2.39 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/jobs/job_skills.csv                  3.34 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/jobs/salaries.csv                    2.15 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/mappings/industries.csv              0.01 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - linkedin-job-postings/mappings/skills.c

## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialisation de la SparkSession connectée à HDFS et Delta Lake
spark = SparkSession.builder \
    .appName("J1_HDFS_Ingestion_Bronze") \
    .getOrCreate()

# Lecture des 3 paramètres HDFS
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a3d7f0d3-c3e5-40f5-afd0-7b3f3009f50a;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central


	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central


	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central


	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central


... (2 outputs left out)

26/11/17 14:43:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    if not fs.exists(dst) or fs.getFileStatus(dst).getLen() != file_path.stat().st_size:
        fs.copyFromLocalFile(False, True, src, dst)
        print(f"Téléversé vers HDFS : {dst}")
    else:
        print(f"Déjà présent sur HDFS : {dst}")

Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/companies/companies.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/companies/company_industries.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/companies/company_specialities.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/companies/employee_counts.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/jobs/benefits.csv


Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/jobs/job_industries.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/jobs/job_skills.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/jobs/salaries.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/mappings/industries.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/mappings/skills.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin-job-postings/postings.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/companies/companies.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/companies/company_industries.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/companies/company_specialities.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/companies/employee_counts.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/jobs/benefi

Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/mappings/skills.csv
Déjà présent sur HDFS : /lakehouse/bronze/raw/linkedin_job_postings/postings.csv


In [4]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : linkedin-job-postings/postings.csv (492.90 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #14 (16.00 Mo) -> DataNode

Fichier : linkedin_job_postings/companies/companies.csv (22.12 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #1 ( 6.12 Mo) -> DataNodes : ['datanode2', 'datanode3']
Fichier : linkedin-job-postings/companies/company_specialities.csv (4.23 Mo | repl=2)
   Bloc #0 ( 4.23 Mo) -> DataNodes : ['datanode1', 'datanode3']
Fichier : linkedin_job_postings/companies/company_specialities.csv (4.23 Mo | repl=2)
   Bloc #0 ( 4.23 Mo) -> DataNodes : ['datanode3', 'datanode1']
Fichier : linkedin-job-postings/jobs/job_skills.csv (3.34 Mo | repl=2)
   Bloc #0 ( 3.34 Mo) -> DataNodes : ['datanode2', 'datanode3']
Fichier : linkedin_job_postings/jobs/job_skills.csv (3.34 Mo | repl=2)
   Bloc #0 ( 3.34 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : linkedin-job-postings/jobs/job_industries.csv (2.39 Mo | repl=2)


   Bloc #0 ( 2.39 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : linkedin_job_postings/jobs/job_industries.csv (2.39 Mo | repl=2)
   Bloc #0 ( 2.39 Mo) -> DataNodes : ['datanode2', 'datanode3']
Fichier : linkedin-job-postings/jobs/salaries.csv (2.15 Mo | repl=2)
   Bloc #0 ( 2.15 Mo) -> DataNodes : ['datanode3', 'datanode1']
Fichier : linkedin_job_postings/jobs/salaries.csv (2.15 Mo | repl=2)
   Bloc #0 ( 2.15 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : linkedin-job-postings/jobs/benefits.csv (1.84 Mo | repl=2)
   Bloc #0 ( 1.84 Mo) -> DataNodes : ['datanode1', 'datanode3']


Fichier : linkedin_job_postings/jobs/benefits.csv (1.84 Mo | repl=2)
   Bloc #0 ( 1.84 Mo) -> DataNodes : ['datanode3', 'datanode2']
Fichier : linkedin-job-postings/companies/employee_counts.csv (1.00 Mo | repl=2)
   Bloc #0 ( 1.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : linkedin_job_postings/companies/employee_counts.csv (1.00 Mo | repl=2)
   Bloc #0 ( 1.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
Fichier : linkedin-job-postings/companies/company_industries.csv (0.75 Mo | repl=2)
   Bloc #0 ( 0.75 Mo) -> DataNodes : ['datanode2', 'datanode3']


Fichier : linkedin_job_postings/companies/company_industries.csv (0.75 Mo | repl=2)
   Bloc #0 ( 0.75 Mo) -> DataNodes : ['datanode2', 'datanode3']
Fichier : linkedin-job-postings/mappings/industries.csv (0.01 Mo | repl=2)
   Bloc #0 ( 0.01 Mo) -> DataNodes : ['datanode1', 'datanode3']
Fichier : linkedin_job_postings/mappings/industries.csv (0.01 Mo | repl=2)
   Bloc #0 ( 0.01 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : linkedin-job-postings/mappings/skills.csv (0.00 Mo | repl=2)
   Bloc #0 ( 0.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
Fichier : linkedin_job_postings/mappings/skills.csv (0.00 Mo | repl=2)
   Bloc #0 ( 0.00 Mo) -> DataNodes : ['datanode1', 'datanode2']


### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [5]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# Lecture du fichier le plus volumineux avec PySpark
df_check = spark.read.option("header", "true").csv(largest_hdfs_path)

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/linkedin-job-postings/postings.csv


Nombre total de lignes lues       : 1,842,544
Nombre de partitions Spark créées : 4


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [6]:
# Définition des tables relationnelles clés du dataset LinkedIn Jobs à ingérer en Bronze :
BRONZE_SOURCES: dict[str, tuple[str, str]] = {
    "postings":    ("linkedin_jobs", "/lakehouse/bronze/raw/linkedin_job_postings/postings.csv"),
    "companies":   ("linkedin_jobs", "/lakehouse/bronze/raw/linkedin_job_postings/companies/companies.csv"),
    "salaries":    ("linkedin_jobs", "/lakehouse/bronze/raw/linkedin_job_postings/jobs/salaries.csv"),
    "job_skills":  ("linkedin_jobs", "/lakehouse/bronze/raw/linkedin_job_postings/jobs/job_skills.csv"),
}

OVERWRITE_EXISTING = False  # Mettre à True pour forcer la ré-ingestion complète

for table_name, source_spec in BRONZE_SOURCES.items():
    dataset_name, raw_path = source_spec if isinstance(source_spec, tuple) else ("default", source_spec)
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"
    delta_log_path = PathHDFS(f"{bronze_delta_path}/_delta_log")

    if not OVERWRITE_EXISTING and fs.exists(delta_log_path):
        cnt = spark.read.format("delta").load(bronze_delta_path).count()
        print(f"[BRONZE OK (validé)] {bronze_delta_path:<38} (source={dataset_name:<18}) -> {cnt:>10,} lignes")
        continue

    # 1. Lecture du CSV brut (sans inferSchema pour rapidité et respect des standards Bronze)
    df_raw = spark.read.option("header", "true").csv(raw_path)

    # 2. Normalisation des noms de colonnes pour compatibilité Delta / Parquet
    clean_cols = [
        re.sub(r"[^a-zA-Z0-9_]+", "_", c.strip().lower()).strip("_")
        for c in df_raw.columns
    ]
    df_raw = df_raw.toDF(*clean_cols)

    # 3. Traçabilité (Date d'ingestion + Dataset source + Fichier source)
    df_bronze = df_raw \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source_dataset", F.lit(dataset_name)) \
        .withColumn("_source_file", F.col("_metadata.file_path"))

    # 4. Écriture en Delta Lake avec écrasement sécurisé
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} (source={dataset_name:<18}) -> {cnt:>10,} lignes")


26/11/17 14:44:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK (validé)] /lakehouse/bronze/postings             (source=linkedin_jobs     ) ->  1,842,544 lignes


26/11/17 14:45:34 WARN NettyRpcEnv: Ignored message: HeartbeatResponse(false)


26/11/17 14:45:38 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Future timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1548)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:484)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala

[BRONZE OK (validé)] /lakehouse/bronze/companies            (source=linkedin_jobs     ) ->     95,251 lignes


[BRONZE OK (validé)] /lakehouse/bronze/salaries             (source=linkedin_jobs     ) ->     40,785 lignes


[BRONZE OK (validé)] /lakehouse/bronze/job_skills           (source=linkedin_jobs     ) ->    213,768 lignes


In [7]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['companies', 'job_skills', 'postings', 'salaries']
